# 🌾 GRAMVIKAS AI - Crop Disease Model Training & Export Notebook
### **Farmer AI Diagnostic Model for Indian Crop Leaf Diseases (Paddy, Tomato, Cotton, Groundnut, Banana, Chilli)**

This notebook trains a lightweight **MobileNetV2 Transfer Learning Model** designed for high accuracy and instant inference in rural settings. The exported `.keras` and `class_names.json` can be loaded directly by the GRAMVIKAS AI backend.

In [ ]:
# Step 1: Install prerequisites & check GPU
!pip install -q tensorflow numpy pillow matplotlib
import tensorflow as tf
print('TensorFlow Version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Step 2: Define Target Indian Crop & Disease Classes
import json
import os

CLASS_NAMES = [
    "Rice___Leaf_Blast",
    "Rice___Brown_Spot",
    "Rice___Bacterial_Leaf_Blight",
    "Rice___Healthy",
    "Tomato___Early_Blight",
    "Tomato___Late_Blight",
    "Tomato___Bacterial_Spot",
    "Tomato___Healthy",
    "Cotton___Bacterial_Blight",
    "Cotton___Healthy",
    "Groundnut___Tikka_Leaf_Spot",
    "Banana___Sigatoka_Leaf_Spot",
    "Chilli___Anthracnose_Fruit_Rot",
    "Corn___Common_Rust"
]

print(f'Configured {len(CLASS_NAMES)} classes for Indian agriculture.')
with open('class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)

In [ ]:
# Step 3: Build MobileNetV2 Transfer Learning Architecture
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
NUM_CLASSES = len(CLASS_NAMES)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze pretrained feature extractor

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1./127.5, offset=-1),
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Step 4: Save Model Artifacts
# After fitting on dataset:
model.save('best_crop_model.keras')
print('Saved model to best_crop_model.keras')

# Optional: Convert to TFLite for ultra-low latency edge devices
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('best_crop_model.tflite', 'wb') as f:
    f.write(tflite_model)
print('Exported best_crop_model.tflite')

In [ ]:
# Step 5: Test Inference Pipeline
import numpy as np
from PIL import Image

# Generate mock leaf image for pipeline verification
dummy_img = Image.new('RGB', (224, 224), color=(34, 139, 34))
arr = np.expand_dims(np.array(dummy_img, dtype=np.float32), axis=0)

preds = model.predict(arr)[0]
pred_idx = np.argmax(preds)
print(f'Top Prediction: {CLASS_NAMES[pred_idx]} ({preds[pred_idx]*100:.2f}%)')